In [1]:
import os
import h5torch
import numpy as np
from pyjaspar import jaspardb
import pandas as pd
from Bio.Seq import Seq
from TFBS_negatives.data import HQ_dataset
from sklearn.metrics import roc_auc_score, accuracy_score, matthews_corrcoef, precision_score, recall_score, average_precision_score
import pandas as pd

out_folder = "/data/home/natant/Negatives/Runs/Review_rerun/MOTIFS"
data_folder = "/data/home/natant/Negatives/Data/Encode690/ENCODE_hg38_subset_101bp_celltypes_ATAC_H5_all_chr copy/"
h5t_files = [f for f in os.listdir(data_folder) if f.endswith('.h5t')]
prot_names = []
for h5t_file in h5t_files:
    file_path = os.path.join(data_folder, h5t_file)
    file = h5torch.File(file_path, 'r')
    
    prot_names.extend(file["0/prot_names"][:].astype(str).tolist())

unique_tfs = np.unique(prot_names)
unique_tfs = unique_tfs[unique_tfs != "ATAC_peak"]

jaspar_connection = """ARID3A_(NB100-279):
ATF1_(06-325):
ATF2_(SC-81188): MA1632.1 MA1632.2
ATF3: MA0605.2 MA0605.3
BHLHE40: MA0464.2 MA0464.3
Bach1_(sc-14700): MA1633.1 MA1633.2
CEBPB_(SC-150): MA0466.1 MA0466.2 MA0466.3 MA0466.4
CEBPD_(SC-636): MA0836.1 MA0836.2 MA0836.3
CREB1_(SC-240): MA0018.1 MA0018.2 MA0018.3 MA0018.4 MA0018.5
CTCF: MA0139.1 MA0139.2 MA1929.1 MA1929.2 MA1930.1 MA1930.2
ELF1_(SC-631): MA0473.1 MA0473.2 MA0473.3 MA0473.4
ELK1_(1277-1): MA0028.1 MA0028.2 MA0028.3
ETS1: MA0098.1 MA0098.3 MA0098.4
Egr-1: MA0162.2 MA0162.3 MA0162.4 MA0162.5
FOSL1_(SC-183): MA0477.1 MA0477.2 MA0477.3
FOSL2: MA0478.1 MA0478.2
FOXA1_(SC-101058): MA0148.1 MA0148.2 MA0148.3 MA0148.4 MA0148.5
FOXM1_(SC-502): UN0802.1
GATA3_(SC-268): MA0037.1 MA0037.2 MA0037.3
HSF1: MA0486.1 MA0486.2
IKZF1_(IkN)_(UCLA): MA1508.1 MA1508.2
IRF3: MA1418.1 MA1418.2
JunD: MA0491.1 MA0491.2 MA0491.3 MA0492.1 MA0492.2
MAZ_(ab85725): MA1522.1 MA1522.2
MEF2A: MA0052.1 MA0052.2 MA0052.3 MA0052.4 MA0052.5
MYBL2_(SC-81192): MA0777.1
MafF_(M8194): MA0495.1 MA0495.2 MA0495.3 MA0495.4
MafK_(ab50322): MA0496.1 MA0496.2 MA0496.3 MA0496.4
Max: MA0058.1 MA0058.2 MA0058.3 MA0058.4
Mxi1_(AF4185): MA1108.1 MA1108.2 MA1108.3
NF-YA: MA0060.1 MA0060.2 MA0060.3 MA0060.4
NF-YB: MA0502.1 MA0502.2 MA0502.3
NFIC_(SC-81335): MA0161.1 MA0161.2 MA0161.3 MA1527.1 MA1527.2
NR2F2_(SC-271940): MA1111.1 MA1111.2
Nrf1: MA0506.1
Pbx3: MA1114.1 MA1114.2
RFX5_(200-401-194): MA0510.1 MA0510.2 MA0510.3
RXRA:
SETDB1:
SIX5:
SP1: MA0079.1 MA0079.2 MA0079.3 MA0079.4 MA0079.5
SRF: MA0083.1 MA0083.2 MA0083.3
STAT5A_(SC-74442):
TBP:
TCF12: MA1648.1 MA1648.2
TCF7L2: MA0523.1 MA0523.2
TEAD4_(SC-101184): MA0809.1 MA0809.2 MA0809.3
USF-1:
USF2: MA0526.1 MA0526.2 MA0526.3 MA0526.4 MA0526.5
YY1_(SC-281): MA0095.1 MA0095.2
ZBTB33: MA0527.1 MA0527.2
ZBTB7A_(SC-34508): MA0750.1 MA0750.2 MA0750.3
ZEB1_(SC-25388): MA0103.2 MA0103.3 MA0103.4
ZNF217:
ZNF274: MA1592.1 MA1592.2
ZZZ3:
Znf143_(16618-1-AP): MA0088.2
"""
jaspar_dict = {}
for line in jaspar_connection.strip().split("\n"):
    if ":" in line:
        tf, matrices = line.split(":", 1)
        jaspar_dict[tf.strip()] = matrices.strip().split() if matrices.strip() else []

matrix_ids = [value for values in jaspar_dict.values() for value in values]

jdb_obj = jaspardb(release='JASPAR2024')
motif_objects = {}
for mid in matrix_ids:
    motif = jdb_obj.fetch_motif_by_id(mid)
    motif.pseudocounts = 0.8
    pssm = motif.pssm  # compute the position-specific scoring matrix (PSSM)
    motif_objects[mid] = pssm

part1 =  ['chr13', 'chr13_KI270838v1_alt', 'chr13_KI270839v1_alt', 'chr13_KI270840v1_alt', 'chr13_KI270841v1_alt', 'chr13_KI270842v1_alt', 'chr13_KI270843v1_alt', 'chr18', 'chr18_GL383567v1_alt', 'chr18_GL383568v1_alt', 'chr18_GL383569v1_alt', 'chr18_GL383570v1_alt', 'chr18_GL383571v1_alt', 'chr18_GL383572v1_alt', 'chr18_KI270863v1_alt', 'chr18_KI270864v1_alt', 'chr18_KI270911v1_alt', 'chr18_KI270912v1_alt', 'chr19', 'chr19_GL000209v2_alt', 'chr19_GL383573v1_alt', 'chr19_GL383574v1_alt', 'chr19_GL383575v2_alt', 'chr19_GL383576v1_alt', 'chr19_GL949746v1_alt', 'chr19_GL949747v2_alt', 'chr19_GL949748v2_alt', 'chr19_GL949749v2_alt', 'chr19_GL949750v2_alt', 'chr19_GL949751v2_alt', 'chr19_GL949752v1_alt', 'chr19_GL949753v2_alt', 'chr19_KI270865v1_alt', 'chr19_KI270866v1_alt', 'chr19_KI270867v1_alt', 'chr19_KI270868v1_alt', 'chr19_KI270882v1_alt', 'chr19_KI270883v1_alt', 'chr19_KI270884v1_alt', 'chr19_KI270885v1_alt', 'chr19_KI270886v1_alt', 'chr19_KI270887v1_alt', 'chr19_KI270888v1_alt', 'chr19_KI270889v1_alt', 'chr19_KI270890v1_alt', 'chr19_KI270891v1_alt', 'chr19_KI270914v1_alt', 'chr19_KI270915v1_alt', 'chr19_KI270916v1_alt', 'chr19_KI270917v1_alt', 'chr19_KI270918v1_alt', 'chr19_KI270919v1_alt', 'chr19_KI270920v1_alt', 'chr19_KI270921v1_alt', 'chr19_KI270922v1_alt', 'chr19_KI270923v1_alt', 'chr19_KI270929v1_alt', 'chr19_KI270930v1_alt', 'chr19_KI270931v1_alt', 'chr19_KI270932v1_alt', 'chr19_KI270933v1_alt', 'chr19_KI270938v1_alt', 'chr20', 'chr20_GL383577v2_alt', 'chr20_KI270869v1_alt', 'chr20_KI270870v1_alt', 'chr20_KI270871v1_alt', 'chr3', 'chr3_GL000221v1_random', 'chr3_GL383526v1_alt', 'chr3_JH636055v2_alt', 'chr3_KI270777v1_alt', 'chr3_KI270778v1_alt', 'chr3_KI270779v1_alt', 'chr3_KI270780v1_alt', 'chr3_KI270781v1_alt', 'chr3_KI270782v1_alt', 'chr3_KI270783v1_alt', 'chr3_KI270784v1_alt', 'chr3_KI270895v1_alt', 'chr3_KI270924v1_alt', 'chr3_KI270934v1_alt', 'chr3_KI270935v1_alt', 'chr3_KI270936v1_alt', 'chr3_KI270937v1_alt', 'chr4', 'chr4_GL000008v2_random', 'chr4_GL000257v2_alt', 'chr4_GL383527v1_alt', 'chr4_GL383528v1_alt', 'chr4_KI270785v1_alt', 'chr4_KI270786v1_alt', 'chr4_KI270787v1_alt', 'chr4_KI270788v1_alt', 'chr4_KI270789v1_alt', 'chr4_KI270790v1_alt', 'chr4_KI270896v1_alt', 'chr4_KI270925v1_alt', 'chr7', 'chr7_GL383534v2_alt', 'chr7_KI270803v1_alt', 'chr7_KI270804v1_alt', 'chr7_KI270805v1_alt', 'chr7_KI270806v1_alt', 'chr7_KI270807v1_alt', 'chr7_KI270808v1_alt', 'chr7_KI270809v1_alt', 'chr7_KI270899v1_alt', 'chrX', 'chrX_KI270880v1_alt', 'chrX_KI270881v1_alt', 'chrX_KI270913v1_alt']
part2 = ['chr1', 'chr10', 'chr10_GL383545v1_alt', 'chr10_GL383546v1_alt', 'chr10_KI270824v1_alt', 'chr10_KI270825v1_alt', 'chr11', 'chr11_GL383547v1_alt', 'chr11_JH159136v1_alt', 'chr11_JH159137v1_alt', 'chr11_KI270721v1_random', 'chr11_KI270826v1_alt', 'chr11_KI270827v1_alt', 'chr11_KI270829v1_alt', 'chr11_KI270830v1_alt', 'chr11_KI270831v1_alt', 'chr11_KI270832v1_alt', 'chr11_KI270902v1_alt', 'chr11_KI270903v1_alt', 'chr11_KI270927v1_alt', 'chr15', 'chr15_GL383554v1_alt', 'chr15_GL383555v2_alt', 'chr15_KI270727v1_random', 'chr15_KI270848v1_alt', 'chr15_KI270849v1_alt', 'chr15_KI270850v1_alt', 'chr15_KI270851v1_alt', 'chr15_KI270852v1_alt', 'chr15_KI270905v1_alt', 'chr15_KI270906v1_alt', 'chr1_GL383518v1_alt', 'chr1_GL383519v1_alt', 'chr1_GL383520v2_alt', 'chr1_KI270706v1_random', 'chr1_KI270707v1_random', 'chr1_KI270708v1_random', 'chr1_KI270709v1_random', 'chr1_KI270710v1_random', 'chr1_KI270711v1_random', 'chr1_KI270712v1_random', 'chr1_KI270713v1_random', 'chr1_KI270714v1_random', 'chr1_KI270759v1_alt', 'chr1_KI270760v1_alt', 'chr1_KI270761v1_alt', 'chr1_KI270762v1_alt', 'chr1_KI270763v1_alt', 'chr1_KI270764v1_alt', 'chr1_KI270765v1_alt', 'chr1_KI270766v1_alt', 'chr1_KI270892v1_alt', 'chr21', 'chr21_GL383578v2_alt', 'chr21_GL383579v2_alt', 'chr21_GL383580v2_alt', 'chr21_GL383581v2_alt', 'chr21_KI270872v1_alt', 'chr21_KI270873v1_alt', 'chr21_KI270874v1_alt', 'chr22', 'chr22_GL383582v2_alt', 'chr22_GL383583v2_alt', 'chr22_KB663609v1_alt', 'chr22_KI270731v1_random', 'chr22_KI270732v1_random', 'chr22_KI270733v1_random', 'chr22_KI270734v1_random', 'chr22_KI270735v1_random', 'chr22_KI270736v1_random', 'chr22_KI270737v1_random', 'chr22_KI270738v1_random', 'chr22_KI270739v1_random', 'chr22_KI270875v1_alt', 'chr22_KI270876v1_alt', 'chr22_KI270877v1_alt', 'chr22_KI270878v1_alt', 'chr22_KI270879v1_alt', 'chr22_KI270928v1_alt', 'chr9', 'chr9_GL383539v1_alt', 'chr9_GL383540v1_alt', 'chr9_GL383541v1_alt', 'chr9_GL383542v1_alt', 'chr9_KI270717v1_random', 'chr9_KI270718v1_random', 'chr9_KI270719v1_random', 'chr9_KI270720v1_random', 'chr9_KI270823v1_alt', 'chrY', 'chrY_KI270740v1_random']
part3 = ['chr12', 'chr12_GL383549v1_alt', 'chr12_GL383550v2_alt', 'chr12_GL383551v1_alt', 'chr12_GL383552v1_alt', 'chr12_GL383553v2_alt', 'chr12_GL877875v1_alt', 'chr12_GL877876v1_alt', 'chr12_KI270833v1_alt', 'chr12_KI270834v1_alt', 'chr12_KI270835v1_alt', 'chr12_KI270836v1_alt', 'chr12_KI270837v1_alt', 'chr12_KI270904v1_alt', 'chr14', 'chr14_GL000009v2_random', 'chr14_GL000194v1_random', 'chr14_GL000225v1_random', 'chr14_KI270722v1_random', 'chr14_KI270723v1_random', 'chr14_KI270724v1_random', 'chr14_KI270725v1_random', 'chr14_KI270726v1_random', 'chr14_KI270844v1_alt', 'chr14_KI270845v1_alt', 'chr14_KI270846v1_alt', 'chr14_KI270847v1_alt', 'chr16', 'chr16_GL383556v1_alt', 'chr16_GL383557v1_alt', 'chr16_KI270728v1_random', 'chr16_KI270853v1_alt', 'chr16_KI270854v1_alt', 'chr16_KI270855v1_alt', 'chr16_KI270856v1_alt', 'chr17', 'chr17_GL000205v2_random', 'chr17_GL000258v2_alt', 'chr17_GL383563v3_alt', 'chr17_GL383564v2_alt', 'chr17_GL383565v1_alt', 'chr17_GL383566v1_alt', 'chr17_JH159146v1_alt', 'chr17_JH159147v1_alt', 'chr17_JH159148v1_alt', 'chr17_KI270729v1_random', 'chr17_KI270730v1_random', 'chr17_KI270857v1_alt', 'chr17_KI270858v1_alt', 'chr17_KI270859v1_alt', 'chr17_KI270860v1_alt', 'chr17_KI270861v1_alt', 'chr17_KI270862v1_alt', 'chr17_KI270907v1_alt', 'chr17_KI270908v1_alt', 'chr17_KI270909v1_alt', 'chr17_KI270910v1_alt', 'chr2', 'chr2_GL383521v1_alt', 'chr2_GL383522v1_alt', 'chr2_GL582966v2_alt', 'chr2_KI270715v1_random', 'chr2_KI270716v1_random', 'chr2_KI270767v1_alt', 'chr2_KI270768v1_alt', 'chr2_KI270769v1_alt', 'chr2_KI270770v1_alt', 'chr2_KI270771v1_alt', 'chr2_KI270772v1_alt', 'chr2_KI270773v1_alt', 'chr2_KI270774v1_alt', 'chr2_KI270775v1_alt', 'chr2_KI270776v1_alt', 'chr2_KI270893v1_alt', 'chr2_KI270894v1_alt', 'chr5', 'chr5_GL000208v1_random', 'chr5_GL339449v2_alt', 'chr5_GL383530v1_alt', 'chr5_GL383531v1_alt', 'chr5_GL383532v1_alt', 'chr5_GL949742v1_alt', 'chr5_KI270791v1_alt', 'chr5_KI270792v1_alt', 'chr5_KI270793v1_alt', 'chr5_KI270794v1_alt', 'chr5_KI270795v1_alt', 'chr5_KI270796v1_alt', 'chr5_KI270897v1_alt', 'chr5_KI270898v1_alt', 'chr6', 'chr6_GL000250v2_alt', 'chr6_GL000251v2_alt', 'chr6_GL000252v2_alt', 'chr6_GL000253v2_alt', 'chr6_GL000254v2_alt', 'chr6_GL000255v2_alt', 'chr6_GL000256v2_alt', 'chr6_GL383533v1_alt', 'chr6_KB021644v2_alt', 'chr6_KI270758v1_alt', 'chr6_KI270797v1_alt', 'chr6_KI270798v1_alt', 'chr6_KI270799v1_alt', 'chr6_KI270800v1_alt', 'chr6_KI270801v1_alt', 'chr6_KI270802v1_alt', 'chr8', 'chr8_KI270810v1_alt', 'chr8_KI270811v1_alt', 'chr8_KI270812v1_alt', 'chr8_KI270813v1_alt', 'chr8_KI270814v1_alt', 'chr8_KI270815v1_alt', 'chr8_KI270816v1_alt', 'chr8_KI270817v1_alt', 'chr8_KI270818v1_alt', 'chr8_KI270819v1_alt', 'chr8_KI270820v1_alt', 'chr8_KI270821v1_alt', 'chr8_KI270822v1_alt', 'chr8_KI270900v1_alt', 'chr8_KI270901v1_alt', 'chr8_KI270926v1_alt']
parts = [part1, part2, part3]

cell_types = ["MCF-7", "K562", "GM12878", "HepG2", "HEK293", "A549"]


/data/home/natant/anaconda3/envs/Negs2/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
cell_types = ["GM12878"]
AUROC_scores = {}
accuracy_scores = {}
precision_scores = {}
mcc_scores = {}
recall_scores = {}
specificity_scores = {}
PRAUC_scores = {}

for celltype in cell_types:
    print("celltype: " + celltype)
    tf_auroc_scores = {}
    tf_accuracy_scores = {}
    for fold in range(3):
        print("fold: " + str(fold))
        file = h5torch.File(data_folder+celltype+".h5t", 'r')
        TF_list = [TF.decode() for TF in file["0/prot_names"][:]]
        TF_list.remove("ATAC_peak")

        results = {}
        true_vals = {}
        for TF in TF_list:
            dataset = HQ_dataset(file, TF, subset=parts[fold])
            print(f"TF: {TF}, length: {dataset.__len__()}")
            matrices = jaspar_dict[TF]
            true_vals[TF] = []
            if matrices == []:
                print(f"No matrices found for {TF}")
                continue
            else:
                print(f"Found {len(matrices)} matrices for {TF}")
                for i in range(30): #! range(dataset.__len__()): 
                    # Scan each PWM on both strands
                    seq = Seq("".join([dataset.rev_mapping[i] for i in dataset.__getitem__(i)["1/DNA_regions"]]))
                    true_vals[TF].append(dataset.__getitem__(i)["central"])
                    
                    best_score = []
                    for mid in matrices:
                        pssm = motif_objects[mid] 
                        # Score forward strand
                        scores_fwd = pssm.calculate(seq)
                        max_fwd = np.nanmax(scores_fwd) if len(scores_fwd)>0 else float('-inf')
                        # Score reverse complement
                        rc_seq = str(Seq(seq).reverse_complement())
                        scores_rev = pssm.calculate(rc_seq)
                        max_rev = np.nanmax(scores_rev) if len(scores_rev)>0 else float('-inf')
                        # Take the best (highest) score
                        best_score.append(np.nanmax([max_fwd, max_rev]))

                    results.setdefault(TF, []).append(np.nanmax(best_score))


        for TF, scores in results.items():
            labels = true_vals[TF]
            scores = np.array(scores)
            labels = np.array(labels)
            
            if len(scores) != len(labels):
                print(f"Length mismatch for {TF}: {len(scores)} vs {len(labels)}")
                accuracy_scores.setdefault(TF, []).append((None, None))
                AUROC_scores.setdefault(TF, []).append((None, None))
                PRAUC_scores.setdefault(TF, []).append((None, None))
                mcc_scores.setdefault(TF, []).append((None, None))
                precision_scores.setdefault(TF, []).append((None, None))
                specificity_scores.setdefault(TF, []).append((None, None))
                recall_scores.setdefault(TF, []).append((None, None))
                continue
            if len(np.unique(labels)) < 2:
                fqdgdfgq
                print(f"Only one class present for {TF}")
                print(f"Labels: {np.unique(labels)}")
                accuracy_scores.setdefault(TF, []).append((None, None))
                AUROC_scores.setdefault(TF, []).append((None, None))
                PRAUC_scores.setdefault(TF, []).append((None, None))
                mcc_scores.setdefault(TF, []).append((None, None))
                precision_scores.setdefault(TF, []).append((None, None))
                specificity_scores.setdefault(TF, []).append((None, None))
                recall_scores.setdefault(TF, []).append((None, None))
                continue
            if len(scores) == 0:
                print(f"No scores for {TF}")
                print(f"Scores: {scores}")
                accuracy_scores.setdefault(TF, []).append((None, None))
                AUROC_scores.setdefault(TF, []).append((None, None))
                PRAUC_scores.setdefault(TF, []).append((None, None))
                mcc_scores.setdefault(TF, []).append((None, None))
                precision_scores.setdefault(TF, []).append((None, None))
                specificity_scores.setdefault(TF, []).append((None, None))
                recall_scores.setdefault(TF, []).append((None, None))
                continue
            if len(labels) == 0:
                print(f"No labels for {TF}")
                print(f"Labels: {labels}")
                accuracy_scores.setdefault(TF, []).append((None, None))
                AUROC_scores.setdefault(TF, []).append((None, None))
                PRAUC_scores.setdefault(TF, []).append((None, None))
                mcc_scores.setdefault(TF, []).append((None, None))
                precision_scores.setdefault(TF, []).append((None, None))
                specificity_scores.setdefault(TF, []).append((None, None))
                recall_scores.setdefault(TF, []).append((None, None))
                continue
            if np.isnan(scores).any():
                print(f"NaN values in scores for {TF}")
                print(f"Scores: {np.unique(scores)}")
                accuracy_scores.setdefault(TF, []).append((None, None))
                AUROC_scores.setdefault(TF, []).append((None, None))
                PRAUC_scores.setdefault(TF, []).append((None, None))
                mcc_scores.setdefault(TF, []).append((None, None))
                precision_scores.setdefault(TF, []).append((None, None))
                specificity_scores.setdefault(TF, []).append((None, None))
                recall_scores.setdefault(TF, []).append((None, None))
                continue
            if np.isnan(labels).any():
                print(f"NaN values in labels for {TF}")
                print(f"Labels: {np.unique(labels)}")
                accuracy_scores.setdefault(TF, []).append((None, None))
                AUROC_scores.setdefault(TF, []).append((None, None))
                PRAUC_scores.setdefault(TF, []).append((None, None))
                mcc_scores.setdefault(TF, []).append((None, None))
                precision_scores.setdefault(TF, []).append((None, None))
                specificity_scores.setdefault(TF, []).append((None, None))
                recall_scores.setdefault(TF, []).append((None, None))
                continue
            if len(np.unique(labels)) == 1:
                print(f"Only one class present in labels for {TF}")
                print(f"Labels: {np.unique(labels)}")
                accuracy_scores.setdefault(TF, []).append((None, None))
                AUROC_scores.setdefault(TF, []).append((None, None))
                PRAUC_scores.setdefault(TF, []).append((None, None))
                mcc_scores.setdefault(TF, []).append((None, None))
                precision_scores.setdefault(TF, []).append((None, None))
                specificity_scores.setdefault(TF, []).append((None, None))
                recall_scores.setdefault(TF, []).append((None, None))
                continue
            
            # Calculate AUROC score

            auc = roc_auc_score(labels, scores)
            AUROC_scores.setdefault(TF, []).append((auc,None))
            pr_auc = average_precision_score(labels, scores)
            PRAUC_scores.setdefault(TF, []).append((pr_auc,None))


            # Evaluate MCC, precision and specificity across multiple thresholds, and report the best
            thresholds = np.arange(0.01, 1, 0.01)
            best_mcc, best_prec, best_spec, best_acc, best_rec = -1, -1, -1, -1, -1
            best_t_mcc, best_t_prec, best_t_spec, best_t_acc, best_t_rec = 0.5, 0.5, 0.5, 0.5, 0.5
            for threshold in thresholds:
                predictions = (scores > threshold).astype(int)
                
                mcc = matthews_corrcoef(labels, predictions)
                prec = precision_score(labels, predictions)
                spec = np.sum((predictions == 0) & (labels == 0)) / np.sum(labels == 0) if np.sum(labels == 0) > 0 else 0
                acc = accuracy_score(labels, predictions)
                rec = recall_score(labels, predictions)

                # Calculate true positives, false positives, true negatives, false negatives
                tp = np.sum((predictions == 1) & (labels == 1))
                fp = np.sum((predictions == 1) & (labels == 0))
                tn = np.sum((predictions == 0) & (labels == 0))
                fn = np.sum((predictions == 0) & (labels == 1))
                
                if mcc > best_mcc:
                    best_mcc, best_t_mcc = mcc, threshold
                if prec > best_prec:
                    best_prec, best_t_prec = prec, threshold
                if spec > best_spec:
                    best_spec, best_t_spec = spec, threshold
                if acc > best_acc:
                    best_acc, best_t_acc = acc, threshold
                if rec > best_rec:
                    best_rec, best_t_rec = rec, threshold
            
            mcc_scores.setdefault(TF, []).append((best_mcc, best_t_mcc))
            accuracy_scores.setdefault(TF, []).append((best_acc, best_t_acc))
            precision_scores.setdefault(TF, []).append((best_prec, best_t_prec))
            specificity_scores.setdefault(TF, []).append((best_spec, best_t_spec))
            recall_scores.setdefault(TF, []).append((best_rec, best_t_rec))



                



        


    # df = pd.DataFrame.from_dict(
    #     {tf: AUROC_scores[tf] + accuracy_scores[tf] for tf in AUROC_scores.keys()},
    #     orient='index',
    #     columns=["AUROC_1", "AUROC_2", "AUROC_3", "Accuracy_1", "Accuracy_2", "Accuracy_3"]
    # )
    # # Ensure the output folder exists
    # os.makedirs(out_folder, exist_ok=True)

    # # Write the dataframe to a CSV file in the specified output folder
    # df.to_csv(os.path.join(out_folder, f"{celltype}.csv"), index=True)



celltype: GM12878
fold: 0
TF: CTCF, length: 107270
Found 6 matrices for CTCF
TF: YY1_(SC-281), length: 106666
Found 2 matrices for YY1_(SC-281)
TF: TBP, length: 105835
No matrices found for TBP
TF: Egr-1, length: 105991
Found 4 matrices for Egr-1
TF: Mxi1_(AF4185), length: 105940
Found 3 matrices for Mxi1_(AF4185)
TF: SRF, length: 105611
Found 3 matrices for SRF
TF: MAZ_(ab85725), length: 106018
Found 2 matrices for MAZ_(ab85725)
TF: ELK1_(1277-1), length: 105365
Found 3 matrices for ELK1_(1277-1)
TF: SIX5, length: 105386
No matrices found for SIX5
TF: USF-1, length: 105801
No matrices found for USF-1
TF: SP1, length: 105822
Found 5 matrices for SP1
TF: RFX5_(200-401-194), length: 105380
Found 3 matrices for RFX5_(200-401-194)
TF: ELF1_(SC-631), length: 106239
Found 4 matrices for ELF1_(SC-631)
TF: ATF2_(SC-81188), length: 106077
Found 2 matrices for ATF2_(SC-81188)
TF: NF-YB, length: 106671
Found 3 matrices for NF-YB
TF: USF2, length: 105556
Found 5 matrices for USF2
TF: Znf143_(16618

NameError: name 'fqdgdfgq' is not defined

In [3]:
labels

array([0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0])

In [26]:
AUROC_scores

{'CTCF': [(0.8888888888888888, None), (1.0, None), (0.9503105590062111, None)],
 'YY1_(SC-281)': [(0.9482758620689654, None),
  (0.6964285714285714, None),
  (0.3793103448275862, None)],
 'Egr-1': [(0.9642857142857143, None),
  (0.8620689655172413, None),
  (None, None)],
 'Mxi1_(AF4185)': [(1.0, None), (0.39285714285714285, None), (None, None)],
 'SRF': [(0.7407407407407407, None), (0.7241379310344828, None), (None, None)],
 'MAZ_(ab85725)': [(None, None), (0.7857142857142857, None), (None, None)],
 'ELK1_(1277-1)': [(1.0, None), (0.9310344827586207, None), (None, None)],
 'SP1': [(0.896551724137931, None),
  (0.6785714285714286, None),
  (0.5172413793103448, None)],
 'RFX5_(200-401-194)': [(None, None),
  (0.2068965517241379, None),
  (None, None)],
 'ELF1_(SC-631)': [(1.0, None), (0.7283950617283951, None), (None, None)],
 'ATF2_(SC-81188)': [(0.24137931034482762, None),
  (1.0, None),
  (0.5178571428571428, None)],
 'NF-YB': [(None, None), (1.0, None), (1.0, None)],
 'USF2': [(None

In [27]:
# Create a comprehensive dataframe with all metrics across all folds
df_all_metrics = pd.DataFrame()

for tf in AUROC_scores.keys():
    row_data = {}
    
    # AUROC scores
    for i, (score, _) in enumerate(AUROC_scores[tf], 1):
        row_data[f'AUROC_{i}'] = score
    
    # PRAUC scores
    for i, (score, _) in enumerate(PRAUC_scores[tf], 1):
        row_data[f'PRAUC_{i}'] = score
    
    # Accuracy scores
    for i, (score, thresh) in enumerate(accuracy_scores[tf], 1):
        row_data[f'Accuracy_{i}'] = score
        row_data[f'Accuracy_threshold_{i}'] = thresh
    
    # MCC scores
    for i, (score, thresh) in enumerate(mcc_scores[tf], 1):
        row_data[f'MCC_{i}'] = score
        row_data[f'MCC_threshold_{i}'] = thresh
    
    # Precision scores
    for i, (score, thresh) in enumerate(precision_scores[tf], 1):
        row_data[f'Precision_{i}'] = score
        row_data[f'Precision_threshold_{i}'] = thresh
    
    # Recall scores
    for i, (score, thresh) in enumerate(recall_scores[tf], 1):
        row_data[f'Recall_{i}'] = score
        row_data[f'Recall_threshold_{i}'] = thresh
    
    # Specificity scores
    for i, (score, thresh) in enumerate(specificity_scores[tf], 1):
        row_data[f'Specificity_{i}'] = score
        row_data[f'Specificity_threshold_{i}'] = thresh
    
    # Add row to dataframe
    df_all_metrics = pd.concat([df_all_metrics, pd.DataFrame([row_data], index=[tf])], axis=0)

df_all_metrics

/tmp/ipykernel_665619/1212055748.py:41: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df_all_metrics = pd.concat([df_all_metrics, pd.DataFrame([row_data], index=[tf])], axis=0)
/tmp/ipykernel_665619/1212055748.py:41: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df_all_metrics = pd.concat([df_all_metrics, pd.DataFrame([row_data], index=[tf])], axis=0)
/tmp/ipykernel_665619/1212055748.py:41: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a fu

,AUROC_1,AUROC_2,AUROC_3,PRAUC_1,PRAUC_2,PRAUC_3,Accuracy_1,Accuracy_threshold_1,Accuracy_2,Accuracy_threshold_2,...,Recall_2,Recall_threshold_2,Recall_3,Recall_threshold_3,Specificity_1,Specificity_threshold_1,Specificity_2,Specificity_threshold_2,Specificity_3,Specificity_threshold_3
CTCF,0.888889,1.000000,0.950311,0.488889,1.000000,0.923810,0.433333,0.93,0.500000,0.83,...,1.0,0.01,1.0,0.01,0.370370,0.93,0.423077,0.83,0.217391,0.88
YY1_(SC-281),0.948276,0.696429,0.379310,0.333333,0.297619,0.043478,0.033333,0.01,0.066667,0.01,...,1.0,0.01,1.0,0.01,0.000000,0.01,0.000000,0.01,0.000000,0.01
Egr-1,0.964286,0.862069,NaN,0.583333,0.200000,NaN,0.500000,0.49,0.300000,0.41,...,1.0,0.01,NaN,NaN,0.464286,0.49,0.275862,0.41,NaN,NaN
Mxi1_(AF4185),1.000000,0.392857,NaN,1.000000,0.079193,NaN,0.366667,0.99,0.266667,0.76,...,1.0,0.01,NaN,NaN,0.344828,0.99,0.214286,0.76,NaN,NaN
SRF,0.740741,0.724138,NaN,0.494444,0.111111,NaN,0.566667,0.94,0.733333,0.63,...,1.0,0.01,NaN,NaN,0.555556,0.94,0.724138,0.63,NaN,NaN
MAZ_(ab85725),NaN,0.785714,NaN,NaN,0.571429,NaN,NaN,NaN,0.566667,0.01,...,0.5,0.01,NaN,NaN,NaN,NaN,0.571429,0.01,NaN,NaN
ELK1_(1277-1),1.000000,0.931034,NaN,1.000000,0.333333,NaN,0.033333,0.01,0.100000,0.01,...,1.0,0.01,NaN,NaN,0.000000,0.01,0.068966,0.01,NaN,NaN
SP1,0.896552,0.678571,0.517241,0.250000,0.162500,0.066667,0.033333,0.01,0.066667,0.01,...,1.0,0.01,1.0,0.01,0.000000,0.01,0.000000,0.01,0.000000,0.01
RFX5_(200-401-194),NaN,0.206897,NaN,NaN,0.041667,NaN,NaN,NaN,0.533333,0.86,...,0.0,0.01,NaN,NaN,NaN,NaN,0.551724,0.86,NaN,NaN
ELF1_(SC-631),1.000000,0.728395,NaN,1.000000,0.327778,NaN,0.166667,0.28,0.366667,0.92,...,1.0,0.01,NaN,NaN,0.107143,0.28,0.296296,0.92,NaN,NaN


In [28]:
# Drop rows where all values are NaN
df_all_metrics_cleaned = df_all_metrics.dropna(how='all')
df_all_metrics_cleaned

,AUROC_1,AUROC_2,AUROC_3,PRAUC_1,PRAUC_2,PRAUC_3,Accuracy_1,Accuracy_threshold_1,Accuracy_2,Accuracy_threshold_2,...,Recall_2,Recall_threshold_2,Recall_3,Recall_threshold_3,Specificity_1,Specificity_threshold_1,Specificity_2,Specificity_threshold_2,Specificity_3,Specificity_threshold_3
CTCF,0.888889,1.000000,0.950311,0.488889,1.000000,0.923810,0.433333,0.93,0.500000,0.83,...,1.0,0.01,1.0,0.01,0.370370,0.93,0.423077,0.83,0.217391,0.88
YY1_(SC-281),0.948276,0.696429,0.379310,0.333333,0.297619,0.043478,0.033333,0.01,0.066667,0.01,...,1.0,0.01,1.0,0.01,0.000000,0.01,0.000000,0.01,0.000000,0.01
Egr-1,0.964286,0.862069,NaN,0.583333,0.200000,NaN,0.500000,0.49,0.300000,0.41,...,1.0,0.01,NaN,NaN,0.464286,0.49,0.275862,0.41,NaN,NaN
Mxi1_(AF4185),1.000000,0.392857,NaN,1.000000,0.079193,NaN,0.366667,0.99,0.266667,0.76,...,1.0,0.01,NaN,NaN,0.344828,0.99,0.214286,0.76,NaN,NaN
SRF,0.740741,0.724138,NaN,0.494444,0.111111,NaN,0.566667,0.94,0.733333,0.63,...,1.0,0.01,NaN,NaN,0.555556,0.94,0.724138,0.63,NaN,NaN
MAZ_(ab85725),NaN,0.785714,NaN,NaN,0.571429,NaN,NaN,NaN,0.566667,0.01,...,0.5,0.01,NaN,NaN,NaN,NaN,0.571429,0.01,NaN,NaN
ELK1_(1277-1),1.000000,0.931034,NaN,1.000000,0.333333,NaN,0.033333,0.01,0.100000,0.01,...,1.0,0.01,NaN,NaN,0.000000,0.01,0.068966,0.01,NaN,NaN
SP1,0.896552,0.678571,0.517241,0.250000,0.162500,0.066667,0.033333,0.01,0.066667,0.01,...,1.0,0.01,1.0,0.01,0.000000,0.01,0.000000,0.01,0.000000,0.01
RFX5_(200-401-194),NaN,0.206897,NaN,NaN,0.041667,NaN,NaN,NaN,0.533333,0.86,...,0.0,0.01,NaN,NaN,NaN,NaN,0.551724,0.86,NaN,NaN
ELF1_(SC-631),1.000000,0.728395,NaN,1.000000,0.327778,NaN,0.166667,0.28,0.366667,0.92,...,1.0,0.01,NaN,NaN,0.107143,0.28,0.296296,0.92,NaN,NaN


In [30]:
import pickle

# Ensure the output folder exists
os.makedirs(out_folder, exist_ok=True)

# Save the cleaned dataframe to pickle file
df_all_metrics_cleaned.to_pickle(os.path.join(out_folder, 'GM12878_MOTIFS.pkl'))

print(f"Dataframe saved to {os.path.join(out_folder, 'GM12878_MOTIFS.pkl')}")


Dataframe saved to /data/home/natant/Negatives/Runs/Review_rerun/MOTIFS/GM12878_MOTIFS.pkl
